In [ ]:
import scanpy as sc
import pandas as pd
from harmony import harmonize
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
sc.set_figure_params(dpi_save=300)

In [ ]:
adata = sc.read_h5ad("resources/sc_mosaic_rounded_embeds_uncert_nosubclusters.h5ad")

In [ ]:
adata

In [ ]:
# remove uncertain cells / keep certain cells
adata = adata[adata.obs['uncertainty']<0.7].copy()

In [ ]:
adata.layers["counts"] = adata.X.copy()
# Normalizing
sc.pp.normalize_total(adata, target_sum=1e4)
# Logarithmize
sc.pp.log1p(adata)

## Myeloids

In [ ]:
adata_myo = adata[adata.obs["cluster_id"].isin(["TAM-MG", "TAM-BDM", "Mono", "DC", "Mast"])].copy()

In [ ]:
adata_myo.obs["cluster_id"].value_counts()

In [ ]:
sc.pp.highly_variable_genes(adata_myo, n_top_genes=2000, subset=False, batch_key="sample_id")
sc.pp.pca(adata_myo, svd_solver="arpack", mask_var="highly_variable")

In [ ]:
adata_myo.obsm["Harmony"] = harmonize(adata_myo.obsm["X_pca"], adata_myo.obs, batch_key="sample_id")

In [ ]:
sc.pp.neighbors(adata_myo, use_rep="Harmony")
sc.tl.umap(adata_myo)

In [ ]:
sc.tl.leiden(adata_myo, resolution = 0.5)

In [ ]:
sc.pl.umap(adata_myo, color= ["leiden", 'cluster_id'])

In [ ]:
mds_subtypes = {
    "E-MDSCs": ['GPNMB','SPP1','MT2A','NUPR1','LGALS1','PLIN2','CSTB','FTL','MT1X','VIM','MT1G','GAPDH','CTSD','FTH1','LGALS3'],
    "M-MDSCs": ['LYZ','VIM','S100A6','VCAN','S100A10','CXCL2','TIMP1','S100A4','EREG','CXCL3','FN1','ANXA1','CD44','LGALS1','S100A9'],
    
    #"HLA-DRs": ['HLA-DRB1','HLA-DQA1','HLA-DPB1','HLA-DPA1','HLA-DRA']
}


level2_marker = {
    "MDSCs": mds_subtypes
}

marker_genes_in_data = {}
for celltype in level2_marker["MDSCs"]:
    marker_genes_in_data[celltype] = list(
        set(level2_marker["MDSCs"][celltype]).intersection(adata_myo.var_names)
    )
    print(celltype,
          len(set(level2_marker["MDSCs"][celltype]).intersection(adata_myo.var_names))
          / len(level2_marker["MDSCs"][celltype]))
    print("not included: ",
          set(level2_marker["MDSCs"][celltype]).difference(adata_myo.var_names))

In [ ]:
sc.pl.dotplot(
    adata_myo,
    groupby="leiden",
    var_names=marker_genes_in_data,
    standard_scale='var',  # standard scale: normalize each gene to range from 0 to 1
    use_raw=False
)

In [ ]:
annotation = {
    "3": "E-MDSCs",
    "6": "M-MDSCs"
}
adata_myo.obs["cell_type_sub_new"] = (adata_myo.obs["leiden"].map(annotation).fillna(adata_myo.obs["cluster_id"]))

In [ ]:
adata_myo.obs["cell_type_sub_new"].value_counts()

In [ ]:
adata_myo.write("resources/adata_myo.h5ad")

In [ ]:
# 1. filter obs to only the two cell‐types
mask = adata_myo.obs["cell_type_sub_new"].isin(["E-MDSCs", "M-MDSCs"])

# 2. pull out the barcode (index) and the annotation
df = (
    adata_myo.obs.loc[mask, ["cell_type_sub_new"]]
    .rename_axis("barcode")                    # move index name to a column name
    .reset_index()                             # turn the index into a column
)

# 3. save to CSV
df.to_csv("E_and_M_MDSCs_newscpoli.csv", index=False)

In [ ]:
adata_myo = sc.read_h5ad("resources/adata_myo.h5ad")

In [ ]:
# only keep baseline samples
baselines = np.array(['HKG001a', 'HKG002a', 'HKG003a', 'HKG004a', 'HKG005a', 'HKG006a',
       'HKG007a', 'HKG008a', 'HKG009a', 'HKG010a', 'HKG011a', 'HKG012a',
       'HKG013a', 'HKG015a', 'HKG016a', 'HKG018a', 'HKG019a', 'HKG020a',
       'HKG021a', 'HKG022a', 'HKG023a', 'HKG024a', 'HKG025a', 'HKG026a',
       'HKG027a', 'HKG028a', 'HKG030a', 'HKG031a', 'HKG032a', 'HKG033a',
       'HKG034a', 'HKG035a', 'HKG037a', 'HKG038a', 'HKG039a', 'HKG040a',
       'HKG041a', 'HKG042a', 'HKG045a', 'HKG046a', 'HKG047a', 'HKG048a',
       'HKG049a', 'HKG050a', 'HKG051a', 'HKG052a', 'HKG053a', 'HKG054a',
       'HKG055a', 'HKG057a', 'HKG058a', 'HKG060a', 'HKG063a', 'HKG064a',
       'HKG065a', 'HKG066a', 'HKG067a', 'HKG068a', 'HKG069a', 'HKG070a',
       'HKG071a', 'HKG072a', 'HKG073a', 'HKG074a', 'HKG075a', 'HKG076a',
       'HKG077a', 'HKG078a', 'HKG080a', 'HKG081a', 'HKG083a', 'HKG086b',
       'HKG087a', 'HKG088a', 'HKG089a', 'HKG092b', 'HKG093a', 'HKG094a',
       'HKG098b', 'HKG099a', 'HKG101a', 'HKG102a', 'HKG103a', 'HKG104a',
       'HKG107a', 'HKG108a', 'HKG110a', 'HKG112a', 'HKG114a'])

In [ ]:
adata_myo = adata_myo[adata_myo.obs['sample_id'].isin(baselines)].copy()

In [ ]:
color_palette = {
    # Glial/Neural-related cells (Blues and Purples)
    'OPC': '#1f77b4',          # Dark blue
    'OPC-like': '#4b9cd3',     # Lighter blue
    'Oligodendrocyte': '#6baed6',  # Light blue
    'Astrocyte': '#9ecae1',    # Very light blue
    'TRAIL+ Astrocytes': '#ff00ff',  # Bright magenta (distinct from Astrocyte)
    'AC-like': '#6a51a3',      # Dark purple
    'NPC-like': '#8c6bb1',     # Medium purple
    'RG': '#bcbddc',           # Light purple
    'Neuron': '#dadaeb',      # Very light purple

    # Immune cells (Reds, Oranges, Pinks, Browns, Yellows)
    'TAM-MG': '#d62728',       # Dark red
    'TAM-BDM': '#e66767',      # Lighter red
    'E-MDSCs': '#8B4513',      # Saddle brown (unchanged, distinct from oranges)
    'M-MDSCs': '#DAA520',      # Mustard yellow (distinct from brown and oranges)
    'Mono': '#fdae6b',         # Light orange
    'Neutrophil': '#fdd0a2',   # Very light orange
    'DC': '#e31a1c',           # Bright red
    'CD4/CD8': '#fc4e2a',      # Red-orange
    'NK': '#feb24c',           # Yellow-orange
    'B cell': '#ffeda0',       # Pale yellow
    'Plasma B': '#f768a1',     # Pink
    'Mast': '#ae017e',         # Dark pink

    # Vascular/Other (Greens)
    'Endothelial': '#31a354',   # Dark green
    'Mural cell': '#74c476',   # Medium green
    'MES-like': '#bae4b3'      # Light green
}

In [ ]:
sc.pl.umap(adata_myo, color= ['cell_type_sub_new'], frameon=False, title= "Myeloid Cells", size = 5, palette=color_palette, save="myos_umap_subcluster_baseline.pdf")

In [ ]:
sc.pl.dotplot(
    adata_myo,
    groupby="cell_type_sub_new",
    var_names=marker_genes_in_data,
    standard_scale='var',  # standard scale: normalize each gene to range from 0 to 1
    use_raw=False,
    save="_myos_baseline.pdf"
)

## TRAIL+ Astrocytes

In [ ]:
astros = adata[adata.obs["cluster_id"].isin(["Astrocyte"])].copy()

In [ ]:
sc.pp.highly_variable_genes(astros, n_top_genes=2000, subset=False, batch_key="sample_id")
sc.pp.pca(astros, svd_solver="arpack", mask_var="highly_variable")

In [ ]:
astros.obsm["Harmony"] = harmonize(astros.obsm["X_pca"], astros.obs, batch_key="sample_id")

In [ ]:
sc.pp.neighbors(astros, use_rep="Harmony")
sc.tl.umap(astros)

In [ ]:
astros

In [ ]:
sc.tl.leiden(astros, resolution = 0.5)

In [ ]:
del astros.raw

In [ ]:
sc.pl.umap(astros, color= ["TNFSF10", "leiden"])

In [ ]:
sc.pl.dotplot(
    astros,
    groupby="leiden",
    var_names='TNFSF10',
    standard_scale=None,  # standard scale: normalize each gene to range from 0 to 1
    use_raw=False
)

In [ ]:
astros7 = astros[astros.obs['leiden']=='8']
sc.pl.umap(astros7, color= ["TNFSF10"])

In [ ]:
annotation = {
    "8": "TRAIL+ Astrocytes"
}
astros.obs["cell_type_sub_new"] = (astros.obs["leiden"].map(annotation).fillna(astros.obs["cluster_id"]))

In [ ]:
astros.write("resources/astros.h5ad")

In [ ]:
# 1. filter obs to only the two cell‐types
mask = astros.obs["cell_type_sub_new"].isin(["TRAIL+ Astrocytes"])

# 2. pull out the barcode (index) and the annotation
df = (
    astros.obs.loc[mask, ["cell_type_sub_new"]]
    .rename_axis("barcode")                    # move index name to a column name
    .reset_index()                             # turn the index into a column
)

# 3. save to CSV
df.to_csv("TRAILAstrocytes_newscpoli.csv", index=False)

In [ ]:
astros = sc.read_h5ad("resources/astros.h5ad")

In [ ]:
# only keep baseline samples
baselines = np.array(['HKG001a', 'HKG002a', 'HKG003a', 'HKG004a', 'HKG005a', 'HKG006a',
       'HKG007a', 'HKG008a', 'HKG009a', 'HKG010a', 'HKG011a', 'HKG012a',
       'HKG013a', 'HKG015a', 'HKG016a', 'HKG018a', 'HKG019a', 'HKG020a',
       'HKG021a', 'HKG022a', 'HKG023a', 'HKG024a', 'HKG025a', 'HKG026a',
       'HKG027a', 'HKG028a', 'HKG030a', 'HKG031a', 'HKG032a', 'HKG033a',
       'HKG034a', 'HKG035a', 'HKG037a', 'HKG038a', 'HKG039a', 'HKG040a',
       'HKG041a', 'HKG042a', 'HKG045a', 'HKG046a', 'HKG047a', 'HKG048a',
       'HKG049a', 'HKG050a', 'HKG051a', 'HKG052a', 'HKG053a', 'HKG054a',
       'HKG055a', 'HKG057a', 'HKG058a', 'HKG060a', 'HKG063a', 'HKG064a',
       'HKG065a', 'HKG066a', 'HKG067a', 'HKG068a', 'HKG069a', 'HKG070a',
       'HKG071a', 'HKG072a', 'HKG073a', 'HKG074a', 'HKG075a', 'HKG076a',
       'HKG077a', 'HKG078a', 'HKG080a', 'HKG081a', 'HKG083a', 'HKG086b',
       'HKG087a', 'HKG088a', 'HKG089a', 'HKG092b', 'HKG093a', 'HKG094a',
       'HKG098b', 'HKG099a', 'HKG101a', 'HKG102a', 'HKG103a', 'HKG104a',
       'HKG107a', 'HKG108a', 'HKG110a', 'HKG112a', 'HKG114a'])

astros = astros[astros.obs['sample_id'].isin(baselines)].copy()

In [ ]:
sc.pl.umap(astros, color= ['cell_type_sub_new'], frameon=False, title= "Astrocytes", size = 5, palette=color_palette, save="astros_baseline.pdf")

In [ ]:
sc.pl.dotplot(
    astros,
    var_names=['TNFSF10'],
    groupby="cell_type_sub_new",
    standard_scale=None,
    use_raw=False,
    mean_only_expressed = True,
    dot_max=0.06,       
    figsize=(2.5, 2.5),    
    save="astros_trail_baselines_dotplot_new.png"
)

In [ ]:
sc.pl.umap(astros[astros.obs['cell_type_sub_new']=='TRAIL+ Astrocytes'], color= ['TNFSF10'], frameon=False, title= "TNFSF10 in TRAIL+ Astrocytes", palette=color_palette, save="astros_trail_baselines.pdf")